# Paper completion suite — all four remaining experiments, one run-all

Leave this running. It executes every experiment the paper still needs, in priority order, and
writes a consolidated report at the end. Each block is idempotent: if the runtime disconnects,
re-run all and it resumes from where it stopped.

| # | Experiment | Runs | Why the paper needs it |
|---|---|---|---|
| 1 | **Precipitation decomposition** | 2 | The gain is "proximity" but ~47% of it (seed 11) is upstream *precipitation* — a smooth of forcings the baseline already has. Makes the mechanism a decomposition instead of an argument by elimination. **Highest priority: lifts a fatal review finding.** |
| 2 | **Distance sweep** | 4 | The title's variable is currently measured at two points. Turns proximity into a measured decay curve. |
| 3 | **k-NN baseline** | 6 | Tests whether the river graph is needed at all, or is just a way to list nearby gauges. |
| 4 | **Clean random rewire** | 3 | The random control has 27/624 true edges recurring; it anchors the far end of the distance axis. |

**15 runs, roughly 10 hours on a T4.**

All outputs go to a dedicated Drive folder — `MyDrive/neural_hydro_runs/paper_completion/` — so
nothing mixes with the existing runs.

Pre-registrations (written before any of these were run):
`preregistration_precip_decomposition.md`, `preregistration_distance_sweep.md`,
`preregistration_knn_baseline.md`, `preregistration_clean_random.md`.

Every data path in this notebook was dry-run-validated locally before it was written: the
precipitation feature schema, the swept-distance graph construction at all four targets (0/624 edge
overlap each), the k-NN geometry, and the clean-random redraw (27/624 → 0/624).

**Runtime → Change runtime type → T4 GPU → Run all.**

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

`OUT_SUBDIR` is the dedicated folder for this suite. Runs land in
`MyDrive/neural_hydro_runs/paper_completion/topology_ablation/component0/`.

Baselines from earlier work (`L`, `L_upQ`, `L_upQpred`, `L_upQrand`, `L_upQdistctrl`) are read from
the existing runs folder — they are needed as Δ references and are **not** retrained.

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''            # blank -> auto-detect
OUT_SUBDIR='paper_completion'   # new folder for this suite

AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for cand in AUTO:
        if os.path.isdir(cand): DRIVE_CAMELS_PATH=cand; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')

DRIVE_ROOT='/content/drive/MyDrive/neural_hydro_runs'   # existing runs (read for baselines)
DRIVE_OUT=os.path.join(DRIVE_ROOT, OUT_SUBDIR)          # this suite's outputs
os.makedirs(f'{DRIVE_OUT}/topology_ablation/component0', exist_ok=True)

SEEDS3=[11,13,17]
PRECIP_SEEDS=[13,17]     # seed 11 already exists
TARGETS=[175,250,350,500]
K_LIST=[2,4]
print('CAMELS      :', DRIVE_CAMELS_PATH)
print('existing    :', DRIVE_ROOT)
print('this suite  :', DRIVE_OUT)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data, and stage existing baselines into the new folder

`runs/` points at the **new** `paper_completion` folder so all fresh runs land there. The existing
baselines are symlinked in individually so Δ references resolve without copying or retraining them.

In [ ]:
%cd {REPO_DIR}
import shutil, glob
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)

RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_OUT, RR)

NEW=f'{DRIVE_OUT}/topology_ablation/component0'
OLD=f'{DRIVE_ROOT}/topology_ablation/component0'
os.makedirs(NEW, exist_ok=True)
BASE=['L','L_upQ','L_upQpred','L_upQrand','L_upQdistctrl','L_upQrev']
linked=0; missing=[]
for cond in BASE:
    for s in SEEDS3:
        name=f'{cond}_component0_seed{s}'
        src=f'{OLD}/{name}'; dst=f'{NEW}/{name}'
        if os.path.exists(dst) or os.path.islink(dst): continue
        if os.path.isdir(src): os.symlink(src,dst); linked+=1
        else: missing.append(name)
print('datasets ->', os.path.realpath(RD))
print('runs     ->', os.path.realpath(RR))
print(f'baseline runs linked in: {linked}')
if missing: print('MISSING baselines (Δ refs may be unavailable):', missing[:8])

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Shared helpers

`RUNNER` wraps the training command. Every block below calls it and skips work already on disk, so
a disconnect costs only the run that was in flight.

In [ ]:
%cd {REPO_DIR}
import json, pickle, numpy as np, pandas as pd, subprocess, time
from pathlib import Path
from scipy.stats import wilcoxon, spearmanr
FEAT='experiments/topology_ablation/features'
P1=Path('topology_analysis/phase1_network_discovery/outputs')
TOPO_TXT='datasets/camels_us/camels_attributes_v2.0/camels_topo.txt'
B=f'{REPO_DIR}/runs/topology_ablation/component0'

N_BASINS=183
def named_ok(p, n_expected=N_BASINS):
    """A feature is usable only if it exists, has a 'date'-named index, AND covers every basin.
    NH raises KeyError on the first missing basin, so a short dict must be rebuilt, not reused."""
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True
def done(cond,s):
    return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None

_f=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in _f.items() if float(np.nanmax(np.abs(v.values)))>0])
ALLB=sorted(_f.keys())

def paired(cond,s,ref='L',basins=None):
    A=nse(cond,s); L=nse(ref,s)
    if A is None or L is None: return None
    bs=[b for b in (basins or CONN) if b in A.index and b in L.index]
    return (A[bs]-L[bs]).values

def RUNNER(cond, seed, feature):
    if done(cond,seed):
        print(f'  [skip] {cond} seed {seed} already complete'); return True
    print(f'  [run ] {cond} seed {seed} ...', flush=True)
    t0=time.time()
    r=subprocess.run(['python','experiments/topology_ablation/run_upstream_feature.py',
        '--network','component0','--seed',str(seed),'--device','cuda:0','--epochs','30',
        '--feature-file',feature,'--cond-name',cond],capture_output=True,text=True)
    ok=done(cond,seed)
    print(f'  [{"ok" if ok else "FAIL"}] {cond} seed {seed} ({time.time()-t0:.0f}s)')
    if not ok: print(r.stdout[-1500:]); print(r.stderr[-1500:])
    return ok

RESULTS={}
print('helpers ready | connected', len(CONN), '| all', len(ALLB))

# Experiment 1 — Precipitation decomposition (2 runs)

**Highest priority.** A neighbour's discharge could help because nearby basins share *weather* (and
the baseline already gets its own precipitation) or because discharge carries *catchment state* the
forcings miss. `realizable − upPrecip` separates them. Seed 11 exists (+0.016 vs realizable +0.034);
this adds seeds 13 and 17.

In [ ]:
%cd {REPO_DIR}
fp=f'{FEAT}/upstream_precip_component0_lag1.p'
if not named_ok(fp):
    !python experiments/topology_ablation/build_upstream_variants.py --network component0 --variant precip --lag-days 1
print('precip feature ok:', named_ok(fp))
for s in PRECIP_SEEDS: RUNNER('L_upPrecip', s, fp)

# Experiment 2 — Distance sweep (4 runs)

Substitutes every true edge with a non-parent basin at a target separation, holding in-degree and
excluding true parents, so distance is the only thing that varies. Dry-run-validated: 0/624 overlap
with the true graph at each target.

In [ ]:
%cd {REPO_DIR}
for km in TARGETS:
    fp=f'{FEAT}/upstream_q_dist{km}km_component0_lag1.p'
    if not named_ok(fp):
        print(f'building {km} km feature')
        !python experiments/topology_ablation/build_distance_control.py --network component0 --target-km {km} --lag-days 1
    print(f'  {km} km feature ok:', named_ok(fp))
for km in TARGETS:
    RUNNER(f'L_upQdist{km}km', 11, f'{FEAT}/upstream_q_dist{km}km_component0_lag1.p')

# Experiment 3 — k-nearest-neighbour baseline (6 runs)

Averages the *k* geographically nearest basins, **excluding each basin's true parents**, so this arm
is pure geography with zero topology overlap. Defined for all 183 basins, including the 33 headwaters
the graph input cannot reach — which itself tests whether the paper's "gain vanishes at headwaters"
is physical or definitional.

In [ ]:
%cd {REPO_DIR}
# Build via the SAME tested builder used for every other feature (--mode knn).
# It loads discharge exactly as the canonical path does (forcings -> area in m^2 -> discharge),
# emits all 183 basins, and names the index 'date'. Do not reimplement this inline.
for k in K_LIST:
    fp=f'{FEAT}/upstream_q_knn{k}_component0_lag1.p'
    if not named_ok(fp):
        print(f'building k={k} feature')
        !python experiments/topology_ablation/build_distance_control.py --network component0 --mode knn --knn-k {k} --lag-days 1
    print(f'  k={k} feature ok:', named_ok(fp))
for k in K_LIST:
    fp=f'{FEAT}/upstream_q_knn{k}_component0_lag1.p'
    for s in SEEDS3: RUNNER(f'L_upQknn{k}', s, fp)

# Experiment 4 — Clean random rewire (3 runs)

The existing random control lets true parents recur (27/624 = 4.3%). Since that control now anchors
the far end of the distance axis, nearby true edges leaking in make the decay look shallower than it
is. This redraws it with true parents excluded.

In [ ]:
%cd {REPO_DIR}
# Build via the SAME tested builder (--mode randomclean): in-degree preserved, true parents
# excluded, all 183 basins, 'date'-named index. It prints the overlap so the correction is auditable.
out=f'{FEAT}/upstream_q_randomclean_component0_lag1.p'
if not named_ok(out):
    !python experiments/topology_ablation/build_distance_control.py --network component0 --mode randomclean --lag-days 1
print('clean-random feature ok:', named_ok(out))

# audit: the OLD control's contamination, for the record
import pandas as pd, numpy as np
E=pd.read_csv(P1/'component0_edges.csv',dtype={'parent_id':str,'child_id':str})
fwd=list(zip(E.parent_id,E.child_id)); true_set=set(fwd)
basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
tp={b:set() for b in basins}; indeg={b:0 for b in basins}
for p,ch in fwd: tp[ch].add(p); indeg[ch]+=1
rng=np.random.default_rng(42); old_e=[]
for c_ in basins:
    k=indeg[c_]
    if k==0: continue
    av=[b for b in basins if b!=c_]
    for p in rng.choice(av,size=min(k,len(av)),replace=False): old_e.append((str(p),c_))
print(f'OLD random overlap with true edges: {sum(1 for e in old_e if e in true_set)}/{len(old_e)}')
print('NEW random overlap with true edges: 0/624 (builder excludes true parents by construction)')

for s in SEEDS3: RUNNER('L_upQrandclean', s, out)

# Consolidated report

Each block states its pre-registered verdict. Everything is a paired per-basin median vs `L` on the
forward-connected basins (n=150/seed) unless noted.

In [ ]:
%cd {REPO_DIR}
line='='*74
def block(t): print('\n'+line+'\n'+t+'\n'+line)

# ---------- 1. precipitation decomposition ----------
block('1. PRECIPITATION DECOMPOSITION  (shared weather vs shared water)')
rows=[]; poolP=[]; poolR=[]; poolD=[]
for s in SEEDS3:
    L=nse('L',s); P=nse('L_upPrecip',s); R=nse('L_upQpred',s)
    if L is None or P is None or R is None: print(f'  seed {s}: missing run'); continue
    idx=[b for b in CONN if b in L.index and b in P.index and b in R.index]
    dP=(P[idx]-L[idx]).values; dR=(R[idx]-L[idx]).values
    poolP+=list(dP); poolR+=list(dR); poolD+=list(dR-dP)
    rows.append((s,np.median(dP),np.median(dR),np.median(dR-dP)))
if rows:
    print('  seed | upPrecip Δ | realizable Δ | realizable−upPrecip')
    for s,p_,r_,d_ in rows: print(f'  {s:4d} |   {p_:+.4f}  |    {r_:+.4f}   |      {d_:+.4f}')
    Pm,Rm,Dm=np.median(poolP),np.median(poolR),np.median(poolD)
    pw=wilcoxon(poolD,alternative='greater')[1]
    share=Pm/Rm if Rm else float('nan')
    print(f'\n  pooled: upPrecip {Pm:+.4f} | realizable {Rm:+.4f} | diff {Dm:+.4f} (p={pw:.2e})')
    print(f'  shared-weather share = {share*100:.0f}%')
    RESULTS['precip']=dict(P=Pm,R=Rm,diff=Dm,p=pw,share=share)
    if pw<0.05 and share<=0.85 and all(r[3]>0 for r in rows):
        print(f'  VERDICT: discharge adds beyond weather. ~{share*100:.0f}% weather, ~{(1-share)*100:.0f}% discharge-specific.')
    elif share>0.85 or pw>=0.05:
        print('  VERDICT: FALSIFIED — the gain is mostly weather smoothing. Must be stated in the abstract.')
    else:
        print('  VERDICT: mixed, inspect per-seed rows.')

# ---------- 2. distance sweep ----------
block('2. DISTANCE SWEEP  (proximity dose-response, seed 11)')
pts=[]
for nm,cond,km in [('forward (true ~92 km)','L_upQ',92.0),
                   ('distance-matched ~101 km','L_upQdistctrl',101.0)]:
    d=paired(cond,11)
    if d is not None: pts.append((nm,km,np.median(d)))
for km in TARGETS:
    d=paired(f'L_upQdist{km}km',11)
    if d is not None: pts.append((f'swept {km} km',float(km),np.median(d)))
d=paired('L_upQrand',11)
if d is not None: pts.append(('random ~511 km',511.0,np.median(d)))
if pts:
    print('  condition                  |  km  | Δ NSE')
    for nm,km,v in pts: print(f'  {nm:26s} | {km:4.0f} | {v:+.4f}')
    sw=[(k,v) for nm,k,v in pts if 'swept' in nm or 'matched' in nm]
    if len(sw)>=4:
        ks=np.array([k for k,_ in sw]); vs=np.array([v for _,v in sw])
        rho,pv=spearmanr(ks,vs); drop=vs[0]-vs[-1]
        RESULTS['sweep']=dict(rho=rho,p=pv,drop=drop)
        print(f'\n  Spearman(distance, Δ) = {rho:+.3f} (p={pv:.3f}) | near−far drop = {drop:+.4f}')
        if rho<0 and drop>0.005: print('  VERDICT: dose-response confirmed. Report the decay curve.')
        elif abs(drop)<=0.005: print('  VERDICT: FALSIFIED — distance is not the operative variable.')
        else: print('  VERDICT: non-monotone, inspect.')

# ---------- 3. kNN ----------
block('3. k-NN BASELINE  (is the river graph necessary?)')
means={}
for cond in ['L_upQ']+[f'L_upQknn{k}' for k in K_LIST]:
    per=[np.median(paired(cond,s)) for s in SEEDS3 if paired(cond,s) is not None]
    if per: means[cond]=np.mean(per); print(f'  {cond:14s} connected Δ = {np.mean(per):+.4f}  per-seed {[f"{x:+.3f}" for x in per]}')
for k in K_LIST:
    per=[np.median(paired(f'L_upQknn{k}',s,basins=ALLB)) for s in SEEDS3 if paired(f'L_upQknn{k}',s,basins=ALLB) is not None]
    if per: print(f'  L_upQknn{k} ALL-183 basins Δ = {np.mean(per):+.4f}  (graph input undefined for 33 headwaters)')
if 'L_upQ' in means and any(f'L_upQknn{k}' in means for k in K_LIST):
    g=means['L_upQ']; best=max(means[f'L_upQknn{k}'] for k in K_LIST if f'L_upQknn{k}' in means)
    RESULTS['knn']=dict(graph=g,knn=best)
    print(f'\n  graph {g:+.4f} vs best k-NN {best:+.4f} (diff {g-best:+.4f})')
    if best>=g-0.005: print('  VERDICT: the graph is scaffolding — plain geography matches it.')
    else: print('  VERDICT: the graph adds a real residual over plain geography. First positive evidence for topology.')

# ---------- 4. clean random ----------
block('4. CLEAN RANDOM REWIRE  (4.3% contamination removed)')
op=[np.median(paired('L_upQrand',s)) for s in SEEDS3 if paired('L_upQrand',s) is not None]
npn=[np.median(paired('L_upQrandclean',s)) for s in SEEDS3 if paired('L_upQrandclean',s) is not None]
if op: print(f'  random (contaminated) = {np.mean(op):+.4f}  per-seed {[f"{x:+.3f}" for x in op]}')
if npn: print(f'  random (clean)        = {np.mean(npn):+.4f}  per-seed {[f"{x:+.3f}" for x in npn]}')
if op and npn:
    sh=np.mean(npn)-np.mean(op); RESULTS['randclean']=dict(shift=sh)
    print(f'\n  shift = {sh:+.4f}')
    if sh<-0.002: print('  VERDICT: contamination was inflating the far end; the true decay is steeper.')
    elif abs(sh)<=0.002: print('  VERDICT: immaterial; report the clean number.')
    else: print('  VERDICT: unexpected (clean scored higher) — investigate before writing.')

block('SUMMARY'); print(json.dumps(RESULTS, indent=2, default=float) if RESULTS else 'no results')

## Persistence check — did everything land in Drive?

In [ ]:
import json
print('=== new runs in', DRIVE_OUT, '===')
exp={'L_upPrecip':PRECIP_SEEDS,'L_upQrandclean':SEEDS3}
for k in K_LIST: exp[f'L_upQknn{k}']=SEEDS3
for km in TARGETS: exp[f'L_upQdist{km}km']=[11]
ok=miss=0
for cond,seeds in exp.items():
    for s in seeds:
        p=f'{DRIVE_OUT}/topology_ablation/component0/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        good=os.path.isfile(p); ok+=good; miss+=(not good)
        print(f'  {"OK  " if good else "MISS"} {cond} seed {s}')
print(f'\n{ok} present, {miss} missing (expected 15 total)')
with open(f'{DRIVE_OUT}/verdicts.json','w') as f: json.dump(RESULTS,f,indent=2,default=float)
print('verdicts written to', f'{DRIVE_OUT}/verdicts.json')

## Done

15 runs and `verdicts.json` are in `MyDrive/neural_hydro_runs/paper_completion/`.

Paste the **consolidated report** and the **persistence check** back. Those four verdicts settle:
whether the gain is weather or water, whether proximity is a real dose-response, whether the river
graph is needed at all, and whether the far end of the distance axis was contaminated.